#1.Імпорт всіх необхідних бібліотек

In [ ]:
import numpy as np
import pandas as pd
import warnings
from pathlib import Path
from sklearn.model_selection import train_test_split
import importlib
import src.evaluation
importlib.reload(src.evaluation)
from src.evaluation import evaluate_model, plot_cm
import src.preprocessing
from src.preprocessing import preprocess_news_data, split_data

In [ ]:
pd.set_option('display.max.rows',130)
pd.set_option('display.max.columns',130)
pd.set_option('float_format', '{:.4f}'.format)
pd.set_option("display.max_colwidth",None)

warnings.filterwarnings('ignore')

#2. Завантаження первинних даних

In [ ]:
#dataset = Path("../data/News_Category_Dataset.zip")
dataset = Path("/content/data/News_Category_Dataset.zip")

df = pd.read_json(dataset, lines=True, compression='zip')


In [ ]:
df.head()

,link,headline,category,short_description,authors,date
0,https://www.huffpost.com/entry/covid-boosters-uptake-us_n_632d719ee4b087fae6feaac9,Over 4 Million Americans Roll Up Sleeves For Omicron-Targeted COVID Boosters,U.S. NEWS,Health experts said it is too early to predict whether demand would match up with the 171 million doses of the new boosters the U.S. ordered for the fall.,"Carla K. Johnson, AP",2022-09-23
1,https://www.huffpost.com/entry/american-airlines-passenger-banned-flight-attendant-punch-justice-department_n_632e25d3e4b0e247890329fe,"American Airlines Flyer Charged, Banned For Life After Punching Flight Attendant On Video",U.S. NEWS,"He was subdued by passengers and crew when he fled to the back of the aircraft after the confrontation, according to the U.S. attorney's office in Los Angeles.",Mary Papenfuss,2022-09-23
2,https://www.huffpost.com/entry/funniest-tweets-cats-dogs-september-17-23_n_632de332e4b0695c1d81dc02,23 Of The Funniest Tweets About Cats And Dogs This Week (Sept. 17-23),COMEDY,"""Until you have a dog you don't understand what could be eaten.""",Elyse Wanshel,2022-09-23
3,https://www.huffpost.com/entry/funniest-parenting-tweets_l_632d7d15e4b0d12b5403e479,The Funniest Tweets From Parents This Week (Sept. 17-23),PARENTING,"""Accidentally put grown-up toothpaste on my toddler’s toothbrush and he screamed like I was cleaning his teeth with a Carolina Reaper dipped in Tabasco sauce.""",Caroline Bologna,2022-09-23
4,https://www.huffpost.com/entry/amy-cooper-loses-discrimination-lawsuit-franklin-templeton_n_632c6463e4b09d8701bd227e,Woman Who Called Cops On Black Bird-Watcher Loses Lawsuit Against Ex-Employer,U.S. NEWS,Amy Cooper accused investment firm Franklin Templeton of unfairly firing her and branding her a racist after video of the Central Park encounter went viral.,Nina Golgowski,2022-09-22


#3. Первинна підготовка даних

###3.1 Очищення датасету, створення текстової ознаки

Видалення зайвих пробілів в ключових текстових полях:


In [ ]:
for col in ["headline", "short_description", "authors"]:
  df[col] = df[col].str.strip()

Нагадаємо, що  на етапі дослідницького аналізу було знайдено наступну кількість повних дублікатів:

In [ ]:
df.duplicated().sum()

np.int64(13)

Також на етапі дослідницького аналізу по ключовим тектовим полям було знайдено наступну кількість дублікатів:

In [ ]:
df.duplicated(subset=["headline", "short_description", "category"]).sum()

np.int64(471)

Видалимо вищезазначені дублікати та створимо об'єднане текстове поле на основі полів headline та description:

In [ ]:
df = preprocess_news_data(df)

Перевіримо, чи існують записи з новим полем у яких значення заповнене пустим текстовим рядком, адже воно не має сенсу для навчання моделі:

In [ ]:
(df["text"] == "").sum()

np.int64(0)

Перевіримо, чи існують дублікати в фінальному наборі даних:

In [ ]:
df.duplicated().sum()

np.int64(0)

Перегляне результат внесених змін:

In [ ]:
df.head()

,text,category
0,Over 4 Million Americans Roll Up Sleeves For Omicron-Targeted COVID Boosters Health experts said it is too early to predict whether demand would match up with the 171 million doses of the new boosters the U.S. ordered for the fall.,U.S. NEWS
1,"American Airlines Flyer Charged, Banned For Life After Punching Flight Attendant On Video He was subdued by passengers and crew when he fled to the back of the aircraft after the confrontation, according to the U.S. attorney's office in Los Angeles.",U.S. NEWS
2,"23 Of The Funniest Tweets About Cats And Dogs This Week (Sept. 17-23) ""Until you have a dog you don't understand what could be eaten.""",COMEDY
3,"The Funniest Tweets From Parents This Week (Sept. 17-23) ""Accidentally put grown-up toothpaste on my toddler’s toothbrush and he screamed like I was cleaning his teeth with a Carolina Reaper dipped in Tabasco sauce.""",PARENTING
4,Woman Who Called Cops On Black Bird-Watcher Loses Lawsuit Against Ex-Employer Amy Cooper accused investment firm Franklin Templeton of unfairly firing her and branding her a racist after video of the Central Park encounter went viral.,U.S. NEWS


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 209051 entries, 0 to 209526
Data columns (total 2 columns):
 #   Column    Non-Null Count   Dtype 
---  ------    --------------   ----- 
 0   text      209051 non-null  object
 1   category  209051 non-null  object
dtypes: object(2)
memory usage: 4.8+ MB


###3.3 Розділення даних

Здійснемо розбиття підготовлених даних на train/validation/test набори.

In [ ]:
X_train, X_valid, X_test, y_train, y_valid, y_test = split_data(df)

In [ ]:
train_df = pd.DataFrame({
    "text": X_train,
    "category": y_train
})

valid_df = pd.DataFrame({
    "text": X_valid,
    "category": y_valid
})

test_df = pd.DataFrame({
    "text": X_test,
    "category": y_test
})

train_df.to_csv(
    "data/train.csv.zip",
    index=False,
    compression="zip"
)

valid_df.to_csv(
    "data/valid.csv.zip",
    index=False,
    compression="zip"
)

test_df.to_csv(
    "data/test.csv.zip",
    index=False,
    compression="zip"
)

print("Дані збережено.")

Дані збережено.
